# 01 Data Understanding

This notebook loads the raw dataset and records an initial summary before any cleaning or feature engineering.

Focus areas:
- dataset size and schema
- column meanings
- data types and value ranges
- early issues or modeling considerations worth flagging

In [1]:
from pathlib import Path
import csv

dataset_path = Path("../data/raw/ElectraHub_Data.csv")
with dataset_path.open(newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    columns = reader.fieldnames

print(f"Dataset path: {dataset_path}")
print(f"Rows: {len(rows)}")
print(f"Columns: {len(columns)}")

Dataset path: ../data/raw/ElectraHub_Data.csv\n
Rows: 3000\n
Columns: 16\n

In [2]:
print("Columns:")
for column in columns:
    print(f"- {column}")

print("\nFirst record:")
print(rows[0])

Columns:\n
- Region\n
- Product_Category\n
- Campaign_Type\n
- Product_Age_Months\n
- Product_Price\n
- Competitor_Price_Index\n
- Advertising_Expenditure\n
- Discount_Percentage\n
- Campaign_Engagement_Score\n
- Inventory_Level\n
- Num_Reviews\n
- Avg_Customer_Rating\n
- Return_Rate\n
- Length_Product_Description\n
- Popularity\n
- Sales\n
\n
First record:\n
{'Region': 'South', 'Product_Category': 'Tablet', 'Campaign_Type': 'Search Ad', 'Product_Age_Months': '19', 'Product_Price': '1904.04', 'Competitor_Price_Index': '1.053', 'Advertising_Expenditure': '425.33', 'Discount_Percentage': '32.95', 'Campaign_Engagement_Score': '31.66', 'Inventory_Level': '1123', 'Num_Reviews': '370', 'Avg_Customer_Rating': '3.89', 'Return_Rate': '2.702', 'Length_Product_Description': '272', 'Popularity': 'Moderate', 'Sales': '12265.79'}\n

## Column Meanings

| Column | Likely meaning |
| --- | --- |
| `Region` | Sales region for the product campaign or transaction record. |
| `Product_Category` | High-level product type (`Mobile` or `Tablet`). |
| `Campaign_Type` | Marketing channel used for the campaign. |
| `Product_Age_Months` | Age of the product in months. |
| `Product_Price` | Selling price of the product. |
| `Competitor_Price_Index` | Relative competitor pricing, likely centered around `1.0`. Values above `1.0` suggest competitors are priced higher relative to the baseline and below `1.0` lower. |
| `Advertising_Expenditure` | Marketing spend associated with the record. |
| `Discount_Percentage` | Discount applied to the product, expressed as a percentage. |
| `Campaign_Engagement_Score` | Aggregated engagement metric for the campaign. |
| `Inventory_Level` | Available stock units. |
| `Num_Reviews` | Count of customer reviews. |
| `Avg_Customer_Rating` | Average customer rating, likely on a 5-point scale. |
| `Return_Rate` | Product return rate, likely a percentage. |
| `Length_Product_Description` | Description length, likely measured in words or tokens. |
| `Popularity` | Ordinal popularity band from `Very Low` to `Very High`. |
| `Sales` | Target variable: sales amount to model with linear regression. |

In [3]:
missing = {column: 0 for column in columns}
for row in rows:
    for column in columns:
        value = row[column]
        if value is None or value.strip() == "":
            missing[column] += 1

print("Missing values by column:")
for column, count in missing.items():
    print(f"- {column}: {count}")

duplicate_rows = len(rows) - len({tuple(row[column] for column in columns) for row in rows})
print(f"\nDuplicate rows: {duplicate_rows}")

Missing values by column:\n
- Region: 0\n
- Product_Category: 0\n
- Campaign_Type: 0\n
- Product_Age_Months: 0\n
- Product_Price: 0\n
- Competitor_Price_Index: 0\n
- Advertising_Expenditure: 0\n
- Discount_Percentage: 0\n
- Campaign_Engagement_Score: 0\n
- Inventory_Level: 0\n
- Num_Reviews: 0\n
- Avg_Customer_Rating: 0\n
- Return_Rate: 0\n
- Length_Product_Description: 0\n
- Popularity: 0\n
- Sales: 0\n
\n
Duplicate rows: 0\n

## Data Types And Value Ranges

| Column | Role | Data type | Range or categories |
| --- | --- | --- | --- |
| `Region` | categorical | string | `North`, `South`, `East`, `West` |
| `Product_Category` | categorical | string | `Mobile`, `Tablet` |
| `Campaign_Type` | categorical | string | `Social Media`, `Search Ad`, `Email`, `Influencer` |
| `Product_Age_Months` | numeric | integer-like | `1` to `36` |
| `Product_Price` | numeric | float | `10.11` to `2000.00` |
| `Competitor_Price_Index` | numeric | float | `0.780` to `1.250` |
| `Advertising_Expenditure` | numeric | float | `325.34` to `820.00` |
| `Discount_Percentage` | numeric | float | `9.25` to `42.00` |
| `Campaign_Engagement_Score` | numeric | float | `5.72` to `99.00` |
| `Inventory_Level` | numeric | integer-like | `50` to `1840` |
| `Num_Reviews` | numeric | integer-like | `5` to `2989` |
| `Avg_Customer_Rating` | numeric | float | `3.57` to `4.42` |
| `Return_Rate` | numeric | float | `1.383` to `4.279` |
| `Length_Product_Description` | numeric | integer-like | `40` to `448` |
| `Popularity` | categorical / ordinal | string | `Very Low`, `Low`, `Moderate`, `High`, `Very High` |
| `Sales` | numeric target | float | `2500.00` to `40000.00` |

In [4]:
import statistics
from collections import Counter

numeric_columns = [
    "Product_Age_Months",
    "Product_Price",
    "Competitor_Price_Index",
    "Advertising_Expenditure",
    "Discount_Percentage",
    "Campaign_Engagement_Score",
    "Inventory_Level",
    "Num_Reviews",
    "Avg_Customer_Rating",
    "Return_Rate",
    "Length_Product_Description",
    "Sales",
]
categorical_columns = ["Region", "Product_Category", "Campaign_Type", "Popularity"]

print("Numeric summary:")
for column in numeric_columns:
    values = [float(row[column]) for row in rows]
    print(
        f"{column}: min={min(values)}, max={max(values)}, mean={round(statistics.mean(values), 2)}, median={round(statistics.median(values), 3)}"
    )

print("\nCategorical distribution:")
for column in categorical_columns:
    counts = Counter(row[column] for row in rows)
    print(f"{column}: unique={len(counts)}, counts={dict(counts.most_common())}")

Numeric summary:\n
Product_Age_Months: min=1.0, max=36.0, mean=11.32, median=9.0\n
Product_Price: min=10.11, max=2000.0, mean=980.84, median=906.94\n
Competitor_Price_Index: min=0.78, max=1.25, mean=1.0, median=0.998\n
Advertising_Expenditure: min=325.34, max=820.0, mean=623.78, median=623.8\n
Discount_Percentage: min=9.25, max=42.0, mean=29.44, median=29.54\n
Campaign_Engagement_Score: min=5.72, max=99.0, mean=52.34, median=52.19\n
Inventory_Level: min=50.0, max=1840.0, mean=821.36, median=822.0\n
Num_Reviews: min=5.0, max=2989.0, mean=418.2, median=346.5\n
Avg_Customer_Rating: min=3.57, max=4.42, mean=3.98, median=3.98\n
Return_Rate: min=1.383, max=4.279, mean=2.77, median=2.755\n
Length_Product_Description: min=40.0, max=448.0, mean=247.31, median=248.0\n
Sales: min=2500.0, max=40000.0, mean=26252.86, median=26164.1\n
\n
Categorical distribution:\n
Region: unique=4, counts={'West': 845, 'East': 815, 'South': 690, 'North': 650}\n
Product_Category: unique=2, counts={'Mobile': 1884, 'T

## Initial Observations

- The dataset is clean at first glance: `3000` rows, `16` columns, no missing values, and no exact duplicate rows.
- `Sales` is a continuous numeric target, so linear regression is appropriate as a baseline model.
- `Popularity` looks ordinal rather than purely nominal, so we should consider ordered encoding instead of one-hot encoding.
- `Product_Category` is somewhat imbalanced (`Mobile` is more common than `Tablet`), though not severely.
- `Popularity` is more skewed: `Very Low` appears only `90` times, while `High` and `Very High` dominate.
- Several features appear to be on very different scales, especially `Product_Price`, `Advertising_Expenditure`, `Inventory_Level`, `Num_Reviews`, and `Sales`. Scaling may help once we build the regression pipeline.
- `Avg_Customer_Rating` has a narrow range (`3.57` to `4.42`), which may limit its standalone variation but it can still be informative in combination with other features.
- `Competitor_Price_Index` is tightly centered around `1.0`, suggesting it represents relative market pricing rather than absolute price.
- Early linear signals worth keeping in mind for later modeling: `Campaign_Engagement_Score`, `Advertising_Expenditure`, and `Avg_Customer_Rating` look plausibly useful, while `Return_Rate` may have an inverse relationship with `Sales`.
- The data looks somewhat synthetic because many ranges are neatly bounded and every column is fully populated. That is not a problem, but we should stay careful about over-interpreting real-world causality.

In [2]:
from pathlib import Path
import csv
import statistics

path = Path("../data/raw/ElectraHub_Data.csv")
with path.open(newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    prices = [float(row["Product_Price"]) for row in reader]

average_price = statistics.mean(prices)
print(average_price)

980.8397166666666


In [5]:
import math

sales_values = [float(row["Sales"]) for row in rows]
sales_mean = statistics.mean(sales_values)

print("Approximate linear correlation with Sales:")
for column in numeric_columns:
    if column == "Sales":
        continue
    values = [float(row[column]) for row in rows]
    value_mean = statistics.mean(values)
    numerator = sum((x - value_mean) * (y - sales_mean) for x, y in zip(values, sales_values))
    denominator = math.sqrt(sum((x - value_mean) ** 2 for x in values) * sum((y - sales_mean) ** 2 for y in sales_values))
    correlation = numerator / denominator
    print(f"{column}: {correlation:.4f}")

Approximate linear correlation with Sales:\n
Product_Age_Months: 0.0395\n
Product_Price: -0.0454\n
Competitor_Price_Index: -0.0775\n
Advertising_Expenditure: 0.6532\n
Discount_Percentage: 0.2745\n
Campaign_Engagement_Score: 0.7223\n
Inventory_Level: 0.0146\n
Num_Reviews: 0.1460\n
Avg_Customer_Rating: 0.4414\n
Return_Rate: -0.1613\n
Length_Product_Description: 0.0143\n